In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

---

In [ ]:
# lab_raw.csv 로드 (구글 드라이브 기준)
# df = pd.read_csv('/content/drive/MyDrive/project/mini_20260109/data/processed/lab_raw.csv')

# lab_raw.csv 로드 (깃허브 기준)
df = pd.read_csv('../data/processed/lab_raw.csv')

In [ ]:
# 시간 컬럼을 날짜 형식으로 변환 (매우 중요)
df['charttime_h'] = pd.to_datetime(df['charttime_h'])

In [ ]:
# 분석 전제 확인용
print("# 행과 열 개수 확인")
print(df.shape)
print("\n# 데이터타입 확인")
print(df.dtypes)

---
## STEP 0. 분석 단위 고정 (Analysis Contract)


In [ ]:
print("[STEP 0] 분석 단위 확인")
print(f"- 전체 데이터 행 수: {len(df):,}")
print(f"- 대상 환자(Stay) 수: {df['stay_id'].nunique():,}")
print(f"- 포함된 변수: {df.columns.tolist()}")

---
## STEP 1. 데이터 시간순 정렬 확인

In [ ]:
print("[STEP 1] 시간 정렬 샘플 확인 (정상 여부)")
# 한 환자의 첫 5시간 데이터를 봅니다.
display(df.sort_values(['stay_id', 'charttime_h']).head(5))

---
## STEP 2: 시간당 측정 빈도(Data Density)

In [ ]:
print("\n[STEP 2] 시간당 데이터 밀도 확인")
stay_counts = df.groupby('stay_id').size()
print(f"- 환자 1인당 평균 관찰 시간: {stay_counts.mean():.1f}시간")

plt.figure(figsize=(10, 4))
sns.histplot(stay_counts, bins=50, kde=True, color='blue')
plt.title("Distribution of Observation Hours per Stay")
plt.xlabel("Hours")
plt.show()

## STEP 3. 주요 수치 요약 및 이상치 (Outlier)

In [ ]:
print("\n[STEP 3] 이상치 점검 (말 안 되는 수치 찾기)")
# 수치형 컬럼만 선택 (stay_id, charttime 제외)
numeric_cols = df.select_dtypes(include=[np.number]).columns.drop('stay_id')
display(df[numeric_cols].describe().T)

# 대표 변수 하나를 골라 시각화 (예: 'hr' 또는 'glucose' 등)
target_col = numeric_cols[0] # 첫 번째 수치 컬럼 자동 선택
plt.figure(figsize=(10, 4))
sns.boxplot(data=df, x=target_col, color='salmon')
plt.title(f"Outlier Check: {target_col}")
plt.show()

---
## STEP 4: 시간 흐름 패턴 확인 (Trend)

In [ ]:
# 1. 데이터 검증 및 샘플 추출
if df.empty or 'stay_id' not in df.columns:
    raise ValueError("데이터프레임이 비어있거나 stay_id 컬럼이 없습니다")

unique_stays = df['stay_id'].dropna().unique()
if len(unique_stays) == 0:
    raise ValueError("유효한 stay_id가 없습니다")

sample_id = unique_stays[0]
sample_df = df[df['stay_id'] == sample_id].sort_values('charttime_h')

# 2. 분석할 컬럼 필터링
vital_cols = ['sao2', 'ph', 'lactate', 'creatinine', 'bilirubin', 'wbc', 'platelets', 'potassium', 'sodium']
available_cols = [col for col in vital_cols if col in sample_df.columns and not sample_df[col].dropna().empty]

if len(available_cols) == 0:
    raise ValueError("시각화할 유효한 데이터가 없습니다")

# 3. 시각화 (squeeze=False로 일관된 axes 처리)
fig, axes = plt.subplots(len(available_cols), 1, figsize=(12, 2 * len(available_cols)),
                         sharex=True, squeeze=False)
axes = axes.flatten()  # 항상 1D 배열로 변환

for i, col in enumerate(available_cols):
    data = sample_df[['charttime_h', col]].dropna()

    axes[i].plot(data['charttime_h'], data[col], marker='o', markersize=4,
                linestyle='-', color='dodgerblue', linewidth=1.5)
    axes[i].set_title(f"Trend: {col}", loc='left', fontsize=10, fontweight='bold')
    axes[i].grid(True, alpha=0.3, linestyle='--')
    axes[i].set_ylabel(col, fontsize=9)

    # 마지막 subplot에만 x축 레이블 추가
    if i == len(available_cols) - 1:
        axes[i].set_xlabel("Hours since Admission (charttime_h)", fontsize=10)

plt.suptitle(f"Patient Stay ID: {sample_id}", fontsize=12, y=0.995)
plt.tight_layout()
plt.show()


---
## STEP 5 & 6. 결측치 비율 및 패턴 (Missing Map)

In [ ]:
print("\n[STEP 5/6] 결측치 비율 및 지도")
missing_pct = df[numeric_cols].isnull().mean() * 100
print(missing_pct.sort_values(ascending=False))

plt.figure(figsize=(12, 6))
# 데이터가 너무 크면 일부만 샘플링하여 시각화
sns.heatmap(df[numeric_cols].iloc[:1000].isnull(), cbar=False, cmap='viridis')
plt.title("Missing Value Map (Yellow = Missing / Purple = Data)")
plt.show()

---
## STEP 8. 결측과 상태의 관계 (Informative Missing)

In [ ]:
print("\n[STEP 8] 결측의 정보성 확인")
# 예: 특정 변수가 비어있는 시간대와 그렇지 않은 시간대의 다른 수치 비교
df['is_missing'] = df[target_col].isnull()
# 두 번째 수치 컬럼이 있다면 그것과 비교
if len(numeric_cols) > 1:
    compare_col = numeric_cols[1]
    comparison = df.groupby('is_missing')[compare_col].mean()
    print(f"'{target_col}' 결측 여부에 따른 '{compare_col}' 평균 차이:")
    print(comparison)

---
## STEP 9. 최종 판정 (Go/No-Go)

In [ ]:
print("[STEP 9] Sliding Window 투입 가능 여부 판단")
min_hours = stay_counts.min()
if min_hours >= 24: # 우리 코호트 기준이 24시간 이상 체류라면
    print(f"✅ PASS: 모든 환자가 최소 {min_hours}시간 이상의 데이터를 보유함.")
else:
    print(f"⚠️ WARNING: 데이터가 {min_hours}시간 미만인 환자가 존재함. 필터링 필요 여부 확인!")

print("\nEDA 검증 완료.")

In [ ]:
# 0. 사전 검증
required_cols = ['stay_id', 'charttime_h']
missing_cols = [col for col in required_cols if col not in df.columns]
if missing_cols:
    raise ValueError(f"필수 컬럼이 누락되었습니다: {missing_cols}")

# 1. 환자별 체류 시간(Duration) 계산
# charttime_h가 datetime일 경우, (최대값 - 최소값)으로 실제 체류 시간을 구합니다.
stay_summary = df.groupby('stay_id')['charttime_h'].agg(['min', 'max', 'count'])
stay_summary['max_hours'] = (stay_summary['max'] - stay_summary['min']).dt.total_seconds() / 3600
stay_summary.columns = ['first_time', 'last_time', 'record_count', 'max_hours']

short_stays = stay_summary[stay_summary['max_hours'] < 24].index

if len(short_stays) > 0:
    print(f"⚠️ {len(short_stays)}명의 환자가 24시간 미만의 데이터를 보유하고 있습니다.")
    print(f"   (전체 환자 수: {len(stay_summary)}명, 비율: {len(short_stays)/len(stay_summary)*100:.1f}%)\n")

    # 체류 시간 통계 (이제 max_hours는 숫자형이므로 정상 작동합니다)
    short_stats = stay_summary.loc[short_stays, 'max_hours']
    print(f"📊 24시간 미만 환자 체류 시간 통계:")
    print(f"   - 평균: {short_stats.mean():.2f}시간")
    print(f"   - 중앙값: {short_stats.median():.2f}시간")
    print(f"   - 최소: {short_stats.min():.2f}시간")
    print(f"   - 최대: {short_stats.max():.2f}시간\n")

    # 2. 컬럼별 유효 데이터 개수 확인 (이하 동일)
    vital_cols = ['sao2', 'ph', 'lactate', 'creatinine', 'bilirubin', 'wbc', 'platelets', 'potassium', 'sodium']
    available_cols = [col for col in vital_cols if col in df.columns]

    if available_cols:
        short_stay_df = df[df['stay_id'].isin(short_stays)]
        missing_report = short_stay_df.groupby('stay_id')[available_cols].count()

        print("📋 [컬럼별 유효 데이터 개수 - 상위 10명]")
        print(missing_report.head(10).to_string())
        print()

        print("📉 [지표별 데이터 부재 환자 통계]")
        summary_data = []
        for col in available_cols:
            no_data_count = (missing_report[col] == 0).sum()
            has_data_count = (missing_report[col] > 0).sum()
            avg_records = missing_report[col].mean()
            summary_data.append({
                '지표': col,
                '데이터 없음': no_data_count,
                '데이터 있음': has_data_count,
                '평균 레코드 수': f"{avg_records:.1f}"
            })

        summary_df = pd.DataFrame(summary_data)
        print(summary_df.to_string(index=False))
else:
    print("✅ 모든 환자가 24시간 이상의 데이터를 충실히 보유하고 있습니다.")


---

## Step 10. Report MD 생성

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# ==============================================================================
# 1. Report 내용 수집
# ==============================================================================

# 기본 정보
n_rows = len(df)
n_stays = df['stay_id'].nunique()
n_cols = len(df.columns)
columns = df.columns.tolist()

# 수치형 컬럼
numeric_cols = ['sao2', 'ph', 'lactate', 'creatinine', 'bilirubin', 'wbc', 'platelets', 'potassium', 'sodium']
available_numeric = [col for col in numeric_cols if col in df.columns]

# 기술 통계
desc_stats = df[available_numeric].describe().round(2)

# 결측치
missing_data = []
for col in available_numeric:
    missing_count = df[col].isna().sum()
    missing_pct = df[col].isna().mean() * 100
    missing_data.append({
        'Column': col,
        'Missing Count': f"{missing_count:,}",
        'Percentage': f"{missing_pct:.2f}%"
    })
missing_df = pd.DataFrame(missing_data)

# 환자별 관찰 시간
stay_summary = df.groupby('stay_id')['charttime_h'].agg(['min', 'max', 'count'])
stay_summary['max_hours'] = (stay_summary['max'] - stay_summary['min']).dt.total_seconds() / 3600
avg_hours = stay_summary['max_hours'].mean()
median_hours = stay_summary['max_hours'].median()

# 24시간 미만 환자
short_stays = stay_summary[stay_summary['max_hours'] < 24]
n_short = len(short_stays)
pct_short = n_short / n_stays * 100

# ==============================================================================
# 2. Markdown Report 생성
# ==============================================================================

report = f"""# MIMIC-IV Lab Data EDA Report

**Analysis Date:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

---

## 1. Dataset Basic Information

- **Total Records:** {n_rows:,}
- **Unique Patients (stay_id):** {n_stays:,}
- **Number of Features:** {n_cols}

### Column Names and Data Types

```
{df.dtypes.to_string()}
```

본 Lab 데이터는 {n_rows:,}개의 시간별 검사 기록으로 구성되며, 
{n_stays:,}명의 ICU 환자에 대한 혈액검사 결과를 포함한다.

## 2. Descriptive Statistics

### Numerical Variables

```
{desc_stats.to_string()}
```

주요 검사 지표들의 분포를 보면:
- **Lactate**: 평균 {desc_stats.loc['mean', 'lactate']:.2f} mmol/L (정상: <2.0)
- **Creatinine**: 평균 {desc_stats.loc['mean', 'creatinine']:.2f} mg/dL
- **pH**: 평균 {desc_stats.loc['mean', 'ph']:.2f} (정상: 7.35-7.45)
- **Platelets**: 평균 {desc_stats.loc['mean', 'platelets']:.0f} K/uL

## 3. Missing Values Analysis

| Column | Missing Count | Percentage |
|--------|---------------|------------|
"""

for _, row in missing_df.iterrows():
    report += f"| {row['Column']} | {row['Missing Count']} | {row['Percentage']} |\n"

report += f"""
### 결측 패턴 해석

Lab 데이터의 결측은 크게 두 가지 원인으로 구분된다:

1. **검사 미시행**: 임상적으로 해당 검사가 필요하지 않았던 경우
2. **시간대별 측정 주기 차이**: Lab 검사는 vitals와 달리 연속 모니터링이 아닌 의사 오더 기반

특히 **sao2 (동맥혈 산소포화도)**와 **bilirubin**의 높은 결측률은 
ABGA나 간기능 검사가 모든 환자에게 routine으로 시행되지 않음을 반영한다.

결측 자체가 "검사 불필요" 또는 "안정적 상태"를 의미할 수 있으므로,
`lactate_missing`, `abga_checked` 같은 플래그 변수로 정보를 보존하는 전략이 적절하다.

## 4. Temporal Coverage Analysis

### 환자별 Lab 데이터 시간 범위

- **평균 관찰 시간:** {avg_hours:.1f}시간
- **중앙값 관찰 시간:** {median_hours:.1f}시간

### 24시간 미만 데이터 보유 환자

- **해당 환자 수:** {n_short:,}명 ({pct_short:.1f}%)
- 이는 Lab 검사 자체의 특성(비연속적, 오더 기반)으로 인한 것으로,
  vitals 데이터와 병합 시 Lab 값은 forward-fill 등의 전략으로 보간 처리 필요

## 5. Clinical Reference Ranges

| 검사 항목 | 정상 범위 | 단위 | 임상적 의의 |
|----------|----------|------|------------|
| pH | 7.35-7.45 | - | 산-염기 균형 |
| Lactate | <2.0 | mmol/L | 조직 관류 상태 |
| Creatinine | 0.7-1.3 (M), 0.6-1.1 (F) | mg/dL | 신기능 |
| WBC | 4.5-11.0 | K/uL | 감염/염증 |
| Platelets | 150-400 | K/uL | 응고 기능 |
| Potassium | 3.5-5.0 | mEq/L | 전해질 |
| Sodium | 136-145 | mEq/L | 전해질 |
| Bilirubin | 0.1-1.2 | mg/dL | 간기능 |

## 6. Preprocessing Recommendations

### 전처리 전략

| 변수 | 결측률 | 권장 전략 | 근거 |
|------|--------|----------|------|
| creatinine, wbc, platelets | 낮음 | FFill(limit=24) → Median | 일단위 검사 |
| potassium, sodium | 낮음 | FFill(limit=12) → Median | 전해질, 변동 빠름 |
| lactate | 높음 | FFill(limit=12) → 정상값(1.2) + Flag | 측정=중증도 |
| sao2 | 매우 높음 | 드랍 (spo2 중복) | ABGA only |
| ph | 높음 | 드랍 → abga_checked 플래그 | ABGA only |
| bilirubin | 높음 | 드랍 | 예측 기여도 낮음 |

### 핵심 포인트

1. **Lactate**: 72% 결측이지만, "측정 자체가 중증도 지표"
   - 쇼크/패혈증 의심 시에만 측정
   - `lactate_missing` 플래그로 정보 보존

2. **ABGA (ph, sao2)**: 높은 결측률
   - 호흡부전/쇼크 의심 시에만 시행
   - `abga_checked` 플래그로 검사 시행 여부 보존

## 7. Key Visualizations

![Lab EDA Visualizations](00_EDA_Lab_Visualizations.png)

---

## 8. Conclusion

Lab 데이터는 vitals와 달리 비연속적, 오더 기반 측정이므로 
결측률이 높지만, 이는 데이터 품질 문제가 아닌 구조적 특성이다.

전처리 시 다음 사항을 고려해야 한다:
- 검사 미시행 = 정보 (측정 여부 자체가 feature)
- Forward-fill에 적절한 limit 설정 (검사 주기 고려)
- 고결측 변수는 플래그로 대체하여 정보 보존
"""

# ==============================================================================
# 3. Markdown 파일 저장
# ==============================================================================
with open('../reports/00_EDA_Lab_Report.md', 'w', encoding='utf-8') as f:
    f.write(report)
print("✓ Report saved: ../reports/00_EDA_Lab_Report.md")

In [ ]:
# ==============================================================================
# 4. Visualization 생성 및 저장
# ==============================================================================
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Lab Data EDA Visualizations', fontsize=14, fontweight='bold')

# 1. 환자별 관찰 시간 분포
ax1 = axes[0, 0]
sns.histplot(stay_summary['max_hours'], bins=50, kde=True, color='steelblue', ax=ax1)
ax1.set_title('Observation Hours per Patient')
ax1.set_xlabel('Hours')
ax1.set_ylabel('Count')

# 2. 결측률 바 차트
ax2 = axes[0, 1]
missing_pcts = df[available_numeric].isna().mean() * 100
colors = ['#e74c3c' if x > 50 else '#f39c12' if x > 20 else '#27ae60' for x in missing_pcts]
bars = ax2.barh(missing_pcts.index, missing_pcts.values, color=colors)
ax2.set_title('Missing Value Percentage by Column')
ax2.set_xlabel('Missing %')
ax2.axvline(x=50, color='red', linestyle='--', alpha=0.5, label='50%')
ax2.legend()

# 3. Lactate 분포
ax3 = axes[0, 2]
lactate_data = df['lactate'].dropna()
sns.histplot(lactate_data, bins=50, kde=True, color='coral', ax=ax3)
ax3.axvline(x=2.0, color='red', linestyle='--', label='Normal Upper (2.0)')
ax3.set_title('Lactate Distribution')
ax3.set_xlabel('Lactate (mmol/L)')
ax3.legend()

# 4. pH 분포
ax4 = axes[1, 0]
ph_data = df['ph'].dropna()
sns.histplot(ph_data, bins=50, kde=True, color='mediumseagreen', ax=ax4)
ax4.axvline(x=7.35, color='red', linestyle='--', alpha=0.7)
ax4.axvline(x=7.45, color='red', linestyle='--', alpha=0.7, label='Normal Range')
ax4.set_title('pH Distribution')
ax4.set_xlabel('pH')
ax4.legend()

# 5. 결측 히트맵 (샘플)
ax5 = axes[1, 1]
sample_size = min(500, len(df))
sample_df = df[available_numeric].iloc[:sample_size]
sns.heatmap(sample_df.isna(), cbar=False, cmap='YlOrRd', ax=ax5)
ax5.set_title(f'Missing Value Pattern (first {sample_size} rows)')
ax5.set_xlabel('Columns')
ax5.set_ylabel('Rows')

# 6. 주요 Lab 값 Boxplot
ax6 = axes[1, 2]
# 스케일 맞추기 위해 정규화
plot_cols = ['lactate', 'creatinine', 'potassium']
plot_data = df[plot_cols].dropna()
plot_data_normalized = (plot_data - plot_data.mean()) / plot_data.std()
plot_data_normalized.boxplot(ax=ax6)
ax6.set_title('Key Lab Values (Normalized)')
ax6.set_ylabel('Z-score')

plt.tight_layout()
plt.savefig('../reports/00_EDA_Lab_Visualizations.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Visualization saved: 00_EDA_Lab_Visualizations.png")

print("\n" + "="*50)
print("Lab EDA Report 생성 완료!")
print("="*50)